In [2]:
import mysql.connector

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = db.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS login_app")
print("Database created successfully")


Database created successfully


In [3]:
db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="login_app"
)

cursor = db.cursor()


In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100) UNIQUE,
    age INT,
    sex VARCHAR(10),
    password VARCHAR(255)
)
""")


In [5]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import tkinter as tk
from tkinter import messagebox
import mysql.connector

# ------------------ DATABASE CONNECTION ------------------ #
db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",   
    database="login_app"
)

cursor = db.cursor()

# ------------------ FUNCTIONS ------------------ #
def register_user():
    name = reg_name.get()
    email = reg_email.get().strip()
    age = reg_age.get()
    sex = reg_sex.get()
    password = reg_pass.get()
    confirm = reg_conf_pass.get()

    if not name or not email or not age or not sex or not password or not confirm:
        messagebox.showwarning("Warning", "All fields are required")
        return

    if not age.isdigit():
        messagebox.showerror("Error", "Age must be a number")
        return

    if len(password) < 6:
        messagebox.showerror("Error", "Password must be at least 6 characters")
        return

    if password != confirm:
        messagebox.showerror("Error", "Passwords do not match")
        return

    try:
        cursor.execute(
            "INSERT INTO users (name, email, age, sex, password) VALUES (%s, %s, %s, %s, %s)",
            (name, email, age, sex, password)
        )
        db.commit()
        messagebox.showinfo("Success", "Registration Successful")

        registration_window.withdraw()
        login_window.deiconify()

    except mysql.connector.errors.IntegrityError:
        messagebox.showerror("Error", "Email already exists")


def login_user():
    email = login_email.get().strip()
    password = login_pass.get()

    cursor.execute(
        "SELECT * FROM users WHERE email=%s AND password=%s",
        (email, password)
    )

    if cursor.fetchone():
        login_window.withdraw()
        congrats_window.deiconify()
    else:
        messagebox.showerror("Error", "Invalid Email or Password")


def restart():
    congrats_window.withdraw()
    login_window.deiconify()


def go_to_register():
    login_window.withdraw()
    registration_window.deiconify()


def go_to_login():
    registration_window.withdraw()
    login_window.deiconify()


def toggle_password(entry, var):
    entry.config(show="" if var.get() else "*")

# ------------------ ROOT ------------------ #
root = tk.Tk()
root.withdraw()

# ------------------ REGISTRATION WINDOW ------------------ #
registration_window = tk.Toplevel()
registration_window.title("Registration")
registration_window.geometry("380x520")
registration_window.configure(bg="#f0f8ff")

reg_name = tk.StringVar()
reg_email = tk.StringVar()
reg_age = tk.StringVar()
reg_sex = tk.StringVar()
reg_pass = tk.StringVar()
reg_conf_pass = tk.StringVar()

tk.Label(registration_window, text="Name", bg="#f0f8ff").pack(pady=5)
tk.Entry(registration_window, textvariable=reg_name).pack()

tk.Label(registration_window, text="Email", bg="#f0f8ff").pack(pady=5)
tk.Entry(registration_window, textvariable=reg_email).pack()

tk.Label(registration_window, text="Age", bg="#f0f8ff").pack(pady=5)
tk.Entry(registration_window, textvariable=reg_age).pack()

tk.Label(registration_window, text="Sex", bg="#f0f8ff").pack(pady=5)
tk.OptionMenu(registration_window, reg_sex, "Male", "Female", "Other").pack()

tk.Label(registration_window, text="Password", bg="#f0f8ff").pack(pady=5)
reg_pass_entry = tk.Entry(registration_window, textvariable=reg_pass, show="*")
reg_pass_entry.pack()

show_pass = tk.BooleanVar()
tk.Checkbutton(
    registration_window, text="Show Password",
    variable=show_pass, bg="#f0f8ff",
    command=lambda: toggle_password(reg_pass_entry, show_pass)
).pack()

tk.Label(registration_window, text="Confirm Password", bg="#f0f8ff").pack(pady=5)
reg_conf_entry = tk.Entry(registration_window, textvariable=reg_conf_pass, show="*")
reg_conf_entry.pack()

tk.Button(registration_window, text="Register", command=register_user).pack(pady=15)
tk.Button(registration_window, text="Back to Login", command=go_to_login).pack()

# ------------------ LOGIN WINDOW ------------------ #
login_window = tk.Toplevel()
login_window.title("Login")
login_window.geometry("350x280")
login_window.withdraw()
login_window.configure(bg="#e6ffe6")

login_email = tk.StringVar()
login_pass = tk.StringVar()

tk.Label(login_window, text="Email", bg="#e6ffe6").pack(pady=10)
tk.Entry(login_window, textvariable=login_email).pack()

tk.Label(login_window, text="Password", bg="#e6ffe6").pack(pady=10)
login_pass_entry = tk.Entry(login_window, textvariable=login_pass, show="*")
login_pass_entry.pack()

show_login = tk.BooleanVar()
tk.Checkbutton(
    login_window, text="Show Password",
    variable=show_login, bg="#e6ffe6",
    command=lambda: toggle_password(login_pass_entry, show_login)
).pack()

tk.Button(login_window, text="Login", command=login_user).pack(pady=10)
tk.Button(login_window, text="Register", command=go_to_register).pack()

# ------------------ SUCCESS WINDOW ------------------ #
congrats_window = tk.Toplevel()
congrats_window.title("Success")
congrats_window.geometry("320x200")
congrats_window.withdraw()
congrats_window.configure(bg="#fffacd")

tk.Label(congrats_window, text="🎉 Login Successful!", font=("Arial", 16), bg="#fffacd").pack(pady=30)
tk.Button(congrats_window, text="Restart", command=restart).pack()

registration_window.mainloop()
